# Kaggle Variant F 40M (100k Pieces)

Strict training flow for Variant F with:
- real GDN and real CfC backend checks,
- automatic ~40M profile selection from candidate configs,
- optional speed parity precheck (F vs E),
- DDP launch on dual T4 when available.

This notebook is intended for strict, fallback-free Variant F runs.


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import os
import subprocess
import sys

REQUIRE_REAL_GDN = True
REQUIRE_REAL_CFC = True

REQUIRED_PIP_PACKAGES = {
    "miditok": "miditok",
    "pretty_midi": "pretty_midi",
    "ncps": "ncps",
    "safetensors": "safetensors",
}

def ensure_runtime_dependencies() -> None:
    missing = [spec for name, spec in REQUIRED_PIP_PACKAGES.items() if importlib.util.find_spec(name) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", *missing])

def find_project_root() -> Path:
    anchor = Path("training/ablation_runner.py")
    probes = [
        Path.cwd().resolve(),
        Path.cwd().resolve().parent,
        Path("/kaggle/working/piano_midi_model"),
        Path("/kaggle/input/magic_midi"),
        Path("/kaggle/input/magic_midi/piano_midi_model"),
        Path("/kaggle/input/magic-midi"),
        Path("/kaggle/input/datasets/chickaboomcmurtrie/magic-midi"),
    ]
    for probe in probes:
        if not probe.exists():
            continue
        for parent in [probe, *probe.parents]:
            if (parent / anchor).exists() and (parent / "training" / "__init__.py").exists():
                return parent
    raise FileNotFoundError(f"Could not locate project root containing {anchor}")

def _ensure_flash_linear_attention() -> None:
    from model.blocks import gdn_block
    if gdn_block.GDN_AVAILABLE:
        return
    for spec in ["flash-linear-attention", "flash-linear-attention==0.4.2"]:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", spec])
            if gdn_block.try_import_fla():
                return
        except Exception:
            pass
    raise RuntimeError("Real GDN kernels unavailable.")

def _ensure_ncps_cfc() -> None:
    from model import cfc_block
    if cfc_block.CFC_AVAILABLE:
        return
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "ncps"])
    importlib.reload(cfc_block)
    if not cfc_block.CFC_AVAILABLE:
        raise RuntimeError("Real CfC backend unavailable.")

ensure_runtime_dependencies()
PROJECT_ROOT = find_project_root()
OUTPUT_BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")

_ensure_flash_linear_attention()
_ensure_ncps_cfc()

from model.blocks import gdn_block
from model import cfc_block
if REQUIRE_REAL_GDN and not gdn_block.GDN_AVAILABLE:
    raise RuntimeError("Strict mode blocked run: real GDN unavailable.")
if REQUIRE_REAL_CFC and not cfc_block.CFC_AVAILABLE:
    raise RuntimeError("Strict mode blocked run: real CfC unavailable.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Output base: {OUTPUT_BASE}")
print(f"GDN_AVAILABLE: {gdn_block.GDN_AVAILABLE} | CFC_AVAILABLE: {cfc_block.CFC_AVAILABLE}")


In [ ]:
import json
import numpy as np
import torch

from data.tokenizer import CustomDeltaTokenizer
from model.variant_f import VariantFConfig, VariantFModel
from training.ablation_runner import _variant_backend_status

MAX_PIECES = 100_000
EPOCHS = 20
SEED = 42
SEED_LENGTH = 512
CONTINUATION_LENGTH = 1536
MAX_SEQUENCE_LENGTH = SEED_LENGTH + CONTINUATION_LENGTH

TARGET_EFFECTIVE_BATCH = 32
BATCH_PER_GPU = 2
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05
WARMUP_STEPS = 0
MIN_LR_RATIO = 0.1
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.05
MAX_GRAD_NORM = 1.0
LOG_EVERY_N_STEPS = 50
SAVE_EVERY_N_STEPS = 1000
SAVE_EVERY_N_EPOCHS = 5
NUM_WORKERS = max(2, (torch.cuda.device_count() if torch.cuda.is_available() else 1) * 2)

RUN_SPEED_PRECHECK = True
MIN_SPEED_RATIO_VS_E = 0.75
SPEED_BENCH_STEPS = 6
SPEED_BENCH_WARMUP = 2

RUN_TRAINING = False
RUN_RESUME_TRAINING = True
RESUME_FROM_CHECKPOINT = ""
RESUME_MODE = "remaining"

VARIANT_E_BENCH_PROFILE = {
    "d_model": 640, "n_layers": 13, "attention_every_n_layers": 2,
    "gdn_inner_ratio": 0.5, "gdn_num_heads": 4, "gqa_num_heads": 8, "gqa_groups": 4,
}

VARIANT_F_PROFILE_CANDIDATES = [
    {"d_model": 832, "n_layers": 10, "harmonic_ratio": 0.40, "temporal_ratio": 0.30, "gdn_inner_ratio": 0.50, "gdn_num_heads": 4, "temporal_cfc_backbone_units": 624, "temporal_cfc_backbone_layers": 2, "structural_num_heads": 8, "structural_gqa_groups": 4, "cross_stream_every_n_layers": 2, "tokens_per_phrase": 8, "memory_size": 64, "theme_memory_heads": 8},
    {"d_model": 768, "n_layers": 14, "harmonic_ratio": 0.40, "temporal_ratio": 0.30, "gdn_inner_ratio": 0.50, "gdn_num_heads": 4, "temporal_cfc_backbone_units": 576, "temporal_cfc_backbone_layers": 2, "structural_num_heads": 8, "structural_gqa_groups": 4, "cross_stream_every_n_layers": 2, "tokens_per_phrase": 8, "memory_size": 64, "theme_memory_heads": 8},
    {"d_model": 768, "n_layers": 13, "harmonic_ratio": 0.40, "temporal_ratio": 0.30, "gdn_inner_ratio": 0.50, "gdn_num_heads": 4, "temporal_cfc_backbone_units": 576, "temporal_cfc_backbone_layers": 2, "structural_num_heads": 8, "structural_gqa_groups": 4, "cross_stream_every_n_layers": 2, "tokens_per_phrase": 8, "memory_size": 64, "theme_memory_heads": 8},
    {"d_model": 896, "n_layers": 9, "harmonic_ratio": 0.40, "temporal_ratio": 0.30, "gdn_inner_ratio": 0.50, "gdn_num_heads": 4, "temporal_cfc_backbone_units": 672, "temporal_cfc_backbone_layers": 2, "structural_num_heads": 8, "structural_gqa_groups": 4, "cross_stream_every_n_layers": 2, "tokens_per_phrase": 8, "memory_size": 64, "theme_memory_heads": 8},
]

PRETOKENIZED_ROOT_CANDIDATES = [
    PROJECT_ROOT / "processed",
    Path("/kaggle/input/datasets/chickaboomcmurtrie/pulse88-tokenize-100k"),
    Path("/kaggle/input/datasets/chickaboomcmurtrie/pulse88-tokenize-100k/tokenized"),
    Path("/kaggle/input/godzilla-tokenized-100k/tokenized"),
    Path("/kaggle/input/godzilla-tokenized-100k"),
]

GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
WILL_USE_DDP = bool(GPU_COUNT > 1)
GLOBAL_BATCH_BASE = max(1, int(BATCH_PER_GPU) * max(1, int(GPU_COUNT)))
GRAD_ACCUM_STEPS = max(1, int(np.ceil(float(TARGET_EFFECTIVE_BATCH) / float(GLOBAL_BATCH_BASE))))

run_tag = f"sub100m_f_40m_{MAX_PIECES // 1000}k_cfc"
OUTPUT_DIR = OUTPUT_BASE / "outputs" / run_tag
SOURCE_MANIFEST_DIR = OUTPUT_DIR / "source_pretokenized"
MANIFEST_OUT = SOURCE_MANIFEST_DIR / "manifest.json"
EPOCH_EXPORT_ROOT = OUTPUT_DIR / "kaggle_model_exports"

def first_existing_path(paths):
    for path in paths:
        if path.exists():
            return path
    return None

def build_manifest_from_npz(npz_root: Path, manifest_path: Path) -> int:
    npz_files = sorted(npz_root.rglob("*.npz"))
    manifest = []
    for npz_path in npz_files:
        try:
            with np.load(npz_path, allow_pickle=False) as pack:
                if "tokens" not in pack.files:
                    continue
                token_len = int(np.asarray(pack["tokens"]).shape[0])
        except Exception:
            continue
        manifest.append({"md5": npz_path.stem, "npz_path": str(npz_path), "length": token_len, "source_path": str(npz_path.parent)})
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return len(manifest)

PRETOKENIZED_ROOT = first_existing_path(PRETOKENIZED_ROOT_CANDIDATES)
if PRETOKENIZED_ROOT is None:
    raise FileNotFoundError("Could not locate tokenized dataset root.")

PRETOKENIZED_MANIFEST = first_existing_path([
    PRETOKENIZED_ROOT / "manifest.json",
    PRETOKENIZED_ROOT / "processed_pretokenized" / "manifest.json",
    PRETOKENIZED_ROOT / "tokenized" / "manifest.json",
    MANIFEST_OUT,
])
if PRETOKENIZED_MANIFEST is None:
    PRETOKENIZED_MANIFEST = MANIFEST_OUT
    build_manifest_from_npz(PRETOKENIZED_ROOT, PRETOKENIZED_MANIFEST)

tokenizer = CustomDeltaTokenizer(include_special_tokens=False)
if int(getattr(tokenizer, "event_size", 1)) != 4:
    raise RuntimeError("Variant F notebook expects event_size=4 tokenizer.")

def build_f(profile: dict):
    model = VariantFModel(VariantFConfig(vocab_size=int(tokenizer.vocab_size), d_model=int(profile["d_model"]), n_layers=int(profile["n_layers"]), max_sequence_length=int(MAX_SEQUENCE_LENGTH), event_size=4, harmonic_ratio=float(profile["harmonic_ratio"]), temporal_ratio=float(profile["temporal_ratio"]), gdn_inner_ratio=float(profile["gdn_inner_ratio"]), gdn_num_heads=int(profile["gdn_num_heads"]), temporal_cfc_backbone_units=int(profile["temporal_cfc_backbone_units"]), temporal_cfc_backbone_layers=int(profile["temporal_cfc_backbone_layers"]), structural_num_heads=int(profile["structural_num_heads"]), structural_gqa_groups=int(profile["structural_gqa_groups"]), cross_stream_every_n_layers=int(profile["cross_stream_every_n_layers"]), tokens_per_phrase=int(profile["tokens_per_phrase"]), memory_size=int(profile["memory_size"]), theme_memory_heads=int(profile["theme_memory_heads"]), use_continuous_time=True, max_time_seconds=1200.0, use_v2_architecture=True))
    params = int(sum(p.numel() for p in model.parameters()))
    backend = _variant_backend_status(model)
    return model, params, backend

target_params = 40_000_000
candidates = []
for profile in VARIANT_F_PROFILE_CANDIDATES:
    model, params, backend = build_f(profile)
    candidates.append((abs(int(params) - target_params), int(params), dict(profile), dict(backend)))
    del model

candidates.sort(key=lambda x: x[0])
SELECTED_PROFILE = None
SELECTED_PARAMS = 0
SELECTED_BACKEND = {}
for _, params, profile, backend in candidates:
    if backend.get("gdn_using_fallback", False):
        continue
    if backend.get("cfc_using_fallback", False):
        continue
    SELECTED_PROFILE = profile
    SELECTED_PARAMS = int(params)
    SELECTED_BACKEND = backend
    break
if SELECTED_PROFILE is None:
    raise RuntimeError("No Variant F candidate passed strict backend checks.")

entry_count = len(json.loads(Path(PRETOKENIZED_MANIFEST).read_text(encoding="utf-8")))
steps_per_epoch = max(1, int(np.ceil((entry_count * 0.9) / max(1, BATCH_PER_GPU * max(1, GPU_COUNT)) / max(1, GRAD_ACCUM_STEPS))))
total_steps = max(1, int(steps_per_epoch * EPOCHS))
WARMUP_STEPS = int(max(1, int(WARMUP_STEPS) if int(WARMUP_STEPS) > 0 else round(float(WARMUP_RATIO) * float(total_steps))))

print(f"Pretokenized root: {PRETOKENIZED_ROOT}")
print(f"Manifest path: {PRETOKENIZED_MANIFEST}")
print(f"Selected Variant F profile: {SELECTED_PROFILE}")
print(f"Selected params: {SELECTED_PARAMS:,}")
print(f"Backend status: {SELECTED_BACKEND}")
print(f"Estimated steps/epoch: {steps_per_epoch:,} | total_steps: {total_steps:,} | warmup_steps: {WARMUP_STEPS:,}")

if RUN_SPEED_PRECHECK:
    benchmark_script = PROJECT_ROOT / "tools" / "benchmark_kaggle_throughput.py"
    base = [sys.executable, str(benchmark_script), "--seq_len", str(int(MAX_SEQUENCE_LENGTH)), "--batch_size", str(int(max(1, BATCH_PER_GPU))), "--steps", str(int(SPEED_BENCH_STEPS)), "--warmup_steps", str(int(SPEED_BENCH_WARMUP))]
    e_cmd = base + ["--variant", "e", "--d_model", str(int(VARIANT_E_BENCH_PROFILE["d_model"])), "--n_layers", str(int(VARIANT_E_BENCH_PROFILE["n_layers"])), "--attention_every_n_layers", str(int(VARIANT_E_BENCH_PROFILE["attention_every_n_layers"])), "--gdn_inner_ratio", str(float(VARIANT_E_BENCH_PROFILE["gdn_inner_ratio"])), "--gdn_num_heads", str(int(VARIANT_E_BENCH_PROFILE["gdn_num_heads"])), "--gqa_num_heads", str(int(VARIANT_E_BENCH_PROFILE["gqa_num_heads"])), "--gqa_groups", str(int(VARIANT_E_BENCH_PROFILE["gqa_groups"]))]
    f_cmd = base + ["--variant", "f", "--d_model", str(int(SELECTED_PROFILE["d_model"])), "--n_layers", str(int(SELECTED_PROFILE["n_layers"])), "--harmonic_ratio", str(float(SELECTED_PROFILE["harmonic_ratio"])), "--temporal_ratio", str(float(SELECTED_PROFILE["temporal_ratio"])), "--gdn_inner_ratio", str(float(SELECTED_PROFILE["gdn_inner_ratio"])), "--gdn_num_heads", str(int(SELECTED_PROFILE["gdn_num_heads"])), "--temporal_cfc_backbone_units", str(int(SELECTED_PROFILE["temporal_cfc_backbone_units"])), "--temporal_cfc_backbone_layers", str(int(SELECTED_PROFILE["temporal_cfc_backbone_layers"])), "--structural_num_heads", str(int(SELECTED_PROFILE["structural_num_heads"])), "--structural_gqa_groups", str(int(SELECTED_PROFILE["structural_gqa_groups"])), "--cross_stream_every_n_layers", str(int(SELECTED_PROFILE["cross_stream_every_n_layers"])), "--tokens_per_phrase", str(int(SELECTED_PROFILE["tokens_per_phrase"])), "--memory_size", str(int(SELECTED_PROFILE["memory_size"])), "--theme_memory_heads", str(int(SELECTED_PROFILE["theme_memory_heads"]))]
    e_out = subprocess.run(e_cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True, check=True).stdout
    f_out = subprocess.run(f_cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True, check=True).stdout
    def _tps(text: str) -> float:
        for line in text.splitlines():
            if "tokens_per_sec:" in line:
                return float(line.split(":", 1)[1].strip())
        raise RuntimeError("tokens_per_sec not found in benchmark output")
    e_tps = _tps(e_out)
    f_tps = _tps(f_out)
    ratio = float(f_tps) / float(max(1e-6, e_tps))
    print(f"Throughput precheck tokens/sec -> E: {e_tps:.2f} | F: {f_tps:.2f} | ratio(F/E): {ratio:.3f}")
    if ratio < float(MIN_SPEED_RATIO_VS_E):
        raise RuntimeError(f"Variant F is too slow relative to E (ratio={ratio:.3f}, threshold={MIN_SPEED_RATIO_VS_E:.3f}).")


In [ ]:
import shutil

CHECKPOINT_FILE_PRIORITY = ["latest_state.pt", "latest.safetensors", "best_state.pt", "best.safetensors"]
CHECKPOINT_ROOT_CANDIDATES = [
    Path("/kaggle/input/models/chickaboomcmurtrie/variant-f-40m-cfc-piano/pytorch/default/1"),
    Path("/kaggle/input/variant-f-40m-cfc-piano"),
    OUTPUT_DIR / "checkpoints",
]

if RUN_RESUME_TRAINING and not str(RESUME_FROM_CHECKPOINT).strip():
    ranked = []
    for root in CHECKPOINT_ROOT_CANDIDATES:
        if not root.exists():
            continue
        for priority, name in enumerate(CHECKPOINT_FILE_PRIORITY):
            hits = sorted(root.rglob(name), key=lambda p: p.stat().st_mtime, reverse=True)
            if hits:
                ranked.append((priority, -float(hits[0].stat().st_mtime), hits[0]))
    if ranked:
        ranked.sort(key=lambda item: (item[0], item[1]))
        RESUME_FROM_CHECKPOINT = str(ranked[0][2].resolve())

ddp_script = PROJECT_ROOT / "training" / "train_variant_f_40m_ddp.py"
if not ddp_script.exists():
    raise FileNotFoundError(f"Missing trainer script: {ddp_script}")

EPOCH_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

if RUN_TRAINING:
    cmd = [
        str(ddp_script),
        "--pretokenized_manifest", str(PRETOKENIZED_MANIFEST),
        "--pretokenized_root", str(PRETOKENIZED_ROOT),
        "--output_dir", str(OUTPUT_DIR),
        "--max_pieces", str(int(MAX_PIECES)),
        "--seed", str(int(SEED)),
        "--seed_length", str(int(SEED_LENGTH)),
        "--continuation_length", str(int(CONTINUATION_LENGTH)),
        "--max_sequence_length", str(int(MAX_SEQUENCE_LENGTH)),
        "--d_model", str(int(SELECTED_PROFILE["d_model"])),
        "--n_layers", str(int(SELECTED_PROFILE["n_layers"])),
        "--harmonic_ratio", str(float(SELECTED_PROFILE["harmonic_ratio"])),
        "--temporal_ratio", str(float(SELECTED_PROFILE["temporal_ratio"])),
        "--gdn_inner_ratio", str(float(SELECTED_PROFILE["gdn_inner_ratio"])),
        "--gdn_num_heads", str(int(SELECTED_PROFILE["gdn_num_heads"])),
        "--temporal_cfc_backbone_units", str(int(SELECTED_PROFILE["temporal_cfc_backbone_units"])),
        "--temporal_cfc_backbone_layers", str(int(SELECTED_PROFILE["temporal_cfc_backbone_layers"])),
        "--structural_num_heads", str(int(SELECTED_PROFILE["structural_num_heads"])),
        "--structural_gqa_groups", str(int(SELECTED_PROFILE["structural_gqa_groups"])),
        "--cross_stream_every_n_layers", str(int(SELECTED_PROFILE["cross_stream_every_n_layers"])),
        "--tokens_per_phrase", str(int(SELECTED_PROFILE["tokens_per_phrase"])),
        "--memory_size", str(int(SELECTED_PROFILE["memory_size"])),
        "--theme_memory_heads", str(int(SELECTED_PROFILE["theme_memory_heads"])),
        "--epochs", str(int(EPOCHS)),
        "--batch_size", str(int(max(1, BATCH_PER_GPU))),
        "--grad_accumulation_steps", str(int(max(1, GRAD_ACCUM_STEPS))),
        "--learning_rate", str(float(LEARNING_RATE)),
        "--warmup_ratio", str(float(WARMUP_RATIO)),
        "--warmup_steps", str(int(WARMUP_STEPS)),
        "--min_lr_ratio", str(float(MIN_LR_RATIO)),
        "--weight_decay", str(float(WEIGHT_DECAY)),
        "--label_smoothing", str(float(LABEL_SMOOTHING)),
        "--max_grad_norm", str(float(MAX_GRAD_NORM)),
        "--num_workers", str(int(NUM_WORKERS)),
        "--log_every_n_steps", str(int(LOG_EVERY_N_STEPS)),
        "--save_every_n_steps", str(int(max(0, SAVE_EVERY_N_STEPS))),
        "--save_every_n_epochs", str(int(max(1, SAVE_EVERY_N_EPOCHS))),
        "--epoch_bundle_root", str(EPOCH_EXPORT_ROOT),
    ]

    if RUN_RESUME_TRAINING and str(RESUME_FROM_CHECKPOINT).strip():
        cmd.extend(["--resume_from_checkpoint", str(RESUME_FROM_CHECKPOINT), "--resume_mode", str(RESUME_MODE)])
    else:
        cmd.append("--no_auto_resume")

    if WILL_USE_DDP:
        torchrun_bin = shutil.which("torchrun")
        if torchrun_bin:
            full_cmd = [torchrun_bin, "--standalone", "--nnodes=1", "--nproc_per_node", str(int(GPU_COUNT)), *cmd]
        else:
            full_cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nnodes=1", "--nproc_per_node", str(int(GPU_COUNT)), *cmd]
    else:
        full_cmd = [sys.executable, *cmd]

    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
    subprocess.run(full_cmd, check=True, cwd=str(PROJECT_ROOT), env=env)

RESULT_PATH = OUTPUT_DIR / "variant_f_40m_ddp_result.json"
if RESULT_PATH.exists():
    payload = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
    result = payload.get("result", {})
    val_loss = result.get("val_loss", []) if isinstance(result, dict) else []
    print(f"Result JSON: {RESULT_PATH}")
    print(f"Profile: {payload.get('profile')}")
    print(f"Params: {result.get('params')}")
    print(f"Backend: {payload.get('backend_status', {})}")
    if isinstance(val_loss, list) and val_loss:
        print(f"Best val loss: {min(float(v) for v in val_loss):.6f}")
        print(f"Last val loss: {float(val_loss[-1]):.6f}")
else:
    print(f"No result yet at {RESULT_PATH}. Set RUN_TRAINING=True to launch.")
